[Énoncés : Examen blanc A — Fondamentaux mélangés](../examen_blanc_A.ipynb)

# Corrigé — Examen blanc A — Fondamentaux mélangés

Pour chaque exercice : l'idée, la complexité, puis une solution commentée avec tous les tests (y compris les cas cachés de la correction). Exécute d'abord la cellule d'outils.

In [ ]:
# Outils de ce notebook : exécute cette cellule en premier.
# - verifier(fonction, cas) lance ta fonction sur chaque cas et affiche ✅ (bon), ❌ (mauvais) ou 💥 (plantage).
# - chrono(fonction, ...) mesure le temps sur une grande entrée (tests de vitesse facultatifs).
# Tu n'as pas besoin de comprendre ce code pour t'en servir.

import copy
import time

def _court(texte, limite=100):
    return texte if len(texte) <= limite else texte[:limite - 3] + "..."

def verifier(fonction, cas, ne_pas_modifier=False):
    reussis = 0
    for arguments, attendu in cas:
        texte_args = _court(", ".join(repr(a) for a in arguments))
        copie = copy.deepcopy(arguments)
        try:
            obtenu = fonction(*copie)
        except Exception as erreur:
            print(f"💥 {fonction.__name__}({texte_args}) a planté : {type(erreur).__name__}: {erreur}")
            continue
        if obtenu != attendu:
            print(f"❌ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}   (attendu : {_court(repr(attendu))})")
        elif ne_pas_modifier and copie != arguments:
            print(f"⚠️ {fonction.__name__}({texte_args}) -> bon résultat, mais ta fonction a modifié son entrée (interdit ici)")
        else:
            reussis += 1
            print(f"✅ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}")
    print(f"\n{reussis}/{len(cas)} tests réussis")

def chrono(fonction, *arguments, attendu=None, limite=1.0):
    debut = time.perf_counter()
    try:
        obtenu = fonction(*arguments)
    except Exception as erreur:
        print(f"💥 {fonction.__name__} a planté : {type(erreur).__name__}: {erreur}")
        return
    duree = time.perf_counter() - debut
    if obtenu is None:
        print("⏳ Ta fonction ne renvoie rien pour l'instant : écris-la d'abord.")
    elif attendu is not None and obtenu != attendu:
        print(f"❌ Résultat faux sur la grande entrée : {_court(repr(obtenu))}   (attendu : {_court(repr(attendu))})")
    elif duree <= limite:
        print(f"✅ {duree:.3f} s : assez rapide")
    else:
        print(f"⚠️ {duree:.1f} s : trop lent (limite : {limite} s). Ta solution n'a probablement pas la complexité attendue.")

## Exercice 49 — Caractères aux positions impaires

**Idée** : le découpage avec un pas, `texte[debut:fin:pas]`. `texte[1::2]` part de l'indice 1 et avance de 2 en 2. Version avec une boucle : parcourir `range(1, len(texte), 2)` et joindre les caractères.

**Complexité** : O(n).

In [ ]:
def positions_impaires(texte: str) -> str:
    return texte[1::2]          # à partir de l'indice 1, un caractère sur deux


verifier(positions_impaires, [
    (('abcdef',), 'bdf'),
    (('',), ''),
    (('a',), ''),
    (('ab',), 'b'),
    (('abcde',), 'bd'),
])

## Exercice 50 — Colonne la plus chargée

**Idée** : pour chaque colonne `j`, additionner `ligne[j]` sur toutes les lignes. Comparaison stricte (`>`) pour garder le plus petit indice en cas d'égalité, et pas de maximum initialisé à 0 (les sommes peuvent être négatives).

**Complexité** : O(lignes × colonnes).

In [ ]:
def colonne_max(grille: list[list[int]]) -> int:
    meilleure = 0
    meilleure_somme = None
    for j in range(len(grille[0])):
        somme = sum(ligne[j] for ligne in grille)      # somme de la colonne j
        if meilleure_somme is None or somme > meilleure_somme:   # > strict : le plus petit indice gagne
            meilleure, meilleure_somme = j, somme
    return meilleure


verifier(colonne_max, [
    (([[1, 2, 1], [3, 0, 5]],), 2),
    (([[5]],), 0),
    (([[1, 1], [1, 1]],), 0),
    (([[0, -1], [-2, 0]],), 1),
    (([[1, 2, 3]],), 2),
    (([[3], [4]],), 0),
])

## Exercice 51 — Codes distincts suffisamment longs

**Idée** : nettoyer chaque code avec `strip()` puis `lower()`, et ranger ceux qui sont assez longs dans un **set**, qui élimine les doublons tout seul. La réponse est la taille du set.

**Complexité** : O(longueur totale des codes).

In [ ]:
def codes_distincts(codes: list[str]) -> int:
    distincts = set()
    for code in codes:
        propre = code.strip().lower()          # nettoyage : espaces aux bords, puis minuscules
        if len(propre) >= 3:
            distincts.add(propre)
    return len(distincts)


verifier(codes_distincts, [
    ((['  ABC ', 'abc', 'Abd', 'a', 'bcd'],), 3),
    (([],), 0),
    ((['ab', '  a  '],), 0),
    ((['Code', 'CODE', 'code '],), 1),
    ((['a b c', ' abc'],), 2),   # les espaces du milieu restent
    ((['   '],), 0),
])

## Exercice 52 — Plus longue portion non décroissante

**Idée** : un seul passage avec un compteur `courant` : la longueur de la portion non décroissante qui se termine à l'élément actuel. Il augmente si l'élément est ≥ au précédent, sinon il repart à 1. On garde le maximum.

**Complexité** : O(n).

In [ ]:
def suite_non_decroissante(nombres: list[int]) -> int:
    if not nombres:
        return 0
    meilleur = courant = 1               # courant = longueur de la portion qui finit ici
    for i in range(1, len(nombres)):
        if nombres[i] >= nombres[i - 1]:
            courant += 1                 # la portion continue
        else:
            courant = 1                  # elle est cassée : on repart de cet élément
        meilleur = max(meilleur, courant)
    return meilleur


verifier(suite_non_decroissante, [
    (([1, 2, 2, 1, 3, 4],), 3),
    (([],), 0),
    (([5],), 1),
    (([3, 2, 1],), 1),
    (([1, 1, 1, 1],), 4),   # égal compte comme non décroissant
    (([1, 3, 2, 4, 5, 6, 0],), 4),
])

## Exercice 53 — Divisibilité d'un très grand nombre

**Idée** : comme dans la division posée à la main. Lire les chiffres de gauche à droite revient à faire `nombre = nombre × 10 + c` ; or le reste de `nombre × 10 + c` ne dépend que du reste de `nombre`. On ne garde donc que le reste, qui ne dépasse jamais `diviseur`.

**Complexité** : O(nombre de chiffres), avec des calculs sur de petits nombres.

In [ ]:
def grand_nombre_divisible(chiffres: list[int], diviseur: int) -> bool:
    reste = 0
    for c in chiffres:
        # ajouter un chiffre à droite revient à faire « nombre × 10 + c » ;
        # on ne garde que le reste de la division, qui reste toujours petit
        reste = (reste * 10 + c) % diviseur
    return reste == 0


verifier(grand_nombre_divisible, [
    (([1, 2, 3, 4, 5], 3), True),
    (([1, 2, 3, 4, 5], 2), False),
    (([0], 7), True),   # 0 est divisible par tout
    (([7], 7), True),
    (([1, 0, 0], 4), True),
    (([9, 9, 9], 1), True),
    (([1, 2, 3, 4, 5, 6, 7, 8, 9, 0], 11), False),
    (([1, 2, 3, 4, 5, 6, 7, 8, 9, 0], 9), True),
])

In [ ]:
# Test de vitesse
grand_nombre = [(i * 7 + 3) % 10 for i in range(100_000)]   # un nombre de 100 000 chiffres
chrono(grand_nombre_divisible, grand_nombre, 7, attendu=True)
chrono(grand_nombre_divisible, grand_nombre, 9, attendu=True)
chrono(grand_nombre_divisible, [9] * 99_999, 11, attendu=False)

---

[Énoncés de l'examen](../examen_blanc_A.ipynb)